<a href="https://colab.research.google.com/github/imaniiz/CariSurg-Portfolio/blob/feat%2Fweek-7-refactor/Notebooks/Week7_Model_Optimisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CARISURG MedTech Pathways | Healthcare AI Programme | Week 7 Assignment

# **AI-Assisted Triage: Model Optimisation**

---

##**About this Notebook**

This notebook extends the Week 6 baseline modelling analysis by evaluating whether a more sophisticated machine-learning model provides enough benefit to justify its additional complexity. A logistic-regression baseline and a random-forest, gradient boosting and MLP classifier are trained and evaluated using the same cleaned Yale EMMLC emergency department dataset, feature set, train-test split and random seed. The complex models additionally use clinically engineered features. The models are compared across predictive performance, training time, inference time and interpretability.

The primary clinical concern remains the identification of ESI Level 1 patients because under-triaging these patients could delay immediate life-saving care.

##Research Question

Does a random-forest, gradient-boosting or MLP classifier provide a clinically meaningful improvement over logistic regression and decision trees and is that improvement sufficient to justify its additional computational and interpretability costs?

## Notebook Sections

Section 1: Environment Setup and Data Loading

Section 2: Feature Selection

Section 3: Reproducing Week 6 Baseline & Feature Engineering

Section 4: Model 1 - Random-Forest Model

Section 5: Model 2 - Gradient-Boosting Model

Section 6: Model 3 - MLP Model

Section 7: Hyperparameter Tuning with Cross-Validation

Section 8: Six-Axis Benchmark

Section 9: Conclusion and Preliminary Recommendation




---
## **Section 1: Environment Setup and Data Loading**

This section imports the Python libraries required to prepare the data, build the baseline classifiers and evaluate their performance.

- Pandas - Used to load and organise the dataset in tabular form
- NumPy - Supports numerical operations and reproducibility settings
- Matplotlib - Used to create visualisations
- Scikit-learn - Provides the tools used to create the required 80/20 train-test split and train the classifiers
- Time - Used to measure model-training and inference times

A fixed random seed of 42 is used wherever an operation includes randomness.

In [2]:
# Importing relevant python libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import time

from time import perf_counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score
)

pd.set_option("display.width", 120)
print("Libraries loaded and reproducibility settings configured ✅")

Libraries loaded and reproduciblity settings configured ✅


In [3]:
# Environment setup
# Reloading cleaned triage dataset from week 5
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

CLEAN_PATH = Path('/content/drive/MyDrive/Carisurg Portfolio/CariSurg_Week5/triage_cleaned_v1.csv')

df = pd.read_csv(CLEAN_PATH)

# Printing the shape and first five rows of the dataset
print(df.shape)
print("Loaded", df.shape[0], "patients and", df.shape[1], "columns.")
df.head()

Mounted at /content/drive
(55121, 225)
Loaded 55121 patients and 225 columns.


,dep_name,esi,age,gender,ethnicity,race,lang,religion,maritalstatus,employstatus,...,cc_vaginaldischarge,cc_vaginalpain,cc_weakness,cc_wheezing,cc_withdrawal-alcohol,cc_woundcheck,cc_woundinfection,cc_woundre-evaluation,cc_wristinjury,cc_wristpain
0,A,4,87.0,Female,Hispanic or Latino,Other,Other,Pentecostal,Widowed,Retired,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,B,2,53.0,Male,Hispanic or Latino,Other,English,Catholic,Significant Other,Disabled,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,A,2,49.0,Female,Non-Hispanic,White or Caucasian,English,Catholic,Married,Full Time,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,A,3,22.0,Female,Hispanic or Latino,White or Caucasian,English,Catholic,Single,Full Time,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,A,2,62.0,Male,Non-Hispanic,White or Caucasian,English,Protestant,Divorced,Not Employed,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


---
## **Section 2: Feature Selection**

This section defines the target variable (Y) that models will predict and selects the patient characteristics available at the time of triage as input features (X). The target variable is Emergency Severity Index (ESI), represented by the 'esi' column in the dataset.

The predictors include triage vital signs and binary chief-complaint indicators. Demographic variables are excluded from the baseline models to reduce the risk of directly learning demographic in historical triage decisions. Administrative variables and outcomes recorded after triage are also excluded.


In [4]:
# Target variable
TARGET = "esi"

# Vital-sign columns measured at the front door:
VITALS = ["triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr",
          "triage_vital_o2", "triage_vital_temp", "triage_glucose"]

# Demographic variables excluded from the baseline model
DEMOGRAPHICS = ["age", "gender", "ethnicity", "race", "lang", "religion",
                "maritalstatus", "employstatus", "insurance_status"]

# Administrative / arrival details:
ADMIN = ["dep_name", "arrivalmode", "arrivalmonth", "arrivalday", "arrivalhour_bin"]

# OUTCOMES of the visit — known only AFTER triage, so they are excluded from the baseline model
LEAKAGE = ["disposition", "previousdispo"]

# Selecting the permitted model features
FEATURES = [c for c in df.columns if c != TARGET and c not in LEAKAGE + ADMIN + DEMOGRAPHICS]

# X contains the clinical features the models will use as predictors
X = df[FEATURES]

# Y contains the correct ESI level the models will learn to predict
y = df[TARGET]


---
## **Section 3: Reproducing Week 6 Baseline & Feature Engineering**

This section recreates the stratified 80/20 train-test split used during Week 6. A fixed random seed of 42 ensures that the logistic-regression, decision-tree and random-forest models are evaluated using the same patients.

The Week 6 logistic-regression and decision-tree baselines are first trained using the original feature set. Clinical feature engineering is then applied identically to the training and testing data so that its effect can be evaluated fairly.

####**3.1 Reproducing the Week 6 Split**

In [5]:
# Reproducing the week 6 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])

Train: 44096 | Test: 11025


####**3.2 Reviewing cc_other**

In [6]:
# Skips safely if your extract doesn't include a cc_other column
if "cc_other" in df.columns:
    cc_cols = [c for c in df.columns if c.startswith("cc_")]
    total = len(df)
    has_other = int(df["cc_other"].sum())
    only_other = int(((df["cc_other"] == 1) & (df[cc_cols].sum(axis=1) == 1)).sum())

    print(f"Patients flagged cc_other: {has_other} of {total} ({has_other/total:.1%})")
    print(f"...and of those, patients whose ONLY complaint is 'other': {only_other}")
    print("\nMean ESI by cc_other flag (does 'other' lean urgent or not?):")
    print(df.groupby("cc_other")["esi"].mean().round(2))
else:
    print("No cc_other column in this sample — skipping. (On the full extract it will run.)")

Patients flagged cc_other: 4491 of 55121 (8.1%)
...and of those, patients whose ONLY complaint is 'other': 3352

Mean ESI by cc_other flag (does 'other' lean urgent or not?):
cc_other
0.0    2.87
1.0    3.01
2.0    3.26
3.0    3.00
Name: esi, dtype: float64


####**3.3 Week 6 Baselines**

In [7]:
# Rebuilding Week 6 logistic regression baseline so we have a bar to clear.
baseline = make_pipeline(StandardScaler(),
                         LogisticRegression(max_iter=1000, random_state=42))
baseline.fit(X_train, y_train)
baseline_f1 = f1_score(y_test, baseline.predict(X_test), average="macro")
print("Baseline (logistic regression) macro-F1:", round(baseline_f1, 3))

Baseline (logistic regression) macro-F1: 0.495


####**3.4 Feature Engineering: Additional Clinical Features**

- Shock Index - An elevated value may indicate circulatory instability
- Pulse Pressure - Difference between systolic and diastolic blood pressure
- SpO2:Respiratory rate ratio
- Estimates average arterial pressure
- Red-flag indicators - Identify tachypnoea, hypoxia and fever
- Red-flag count

In [8]:
import pandas as pd
# Building new clinical features from existing vitals, and applying them to
# BOTH the train and test sets in the same way.

def add_clinical_features(data):
    out = data.copy()

    # Ratios and combinations supplied as examples
    out["shock_index"]    = out["triage_vital_hr"] / out["triage_vital_sbp"]       # HR / SBP         (uses BP)

    # Adding red flags
    # 1) Mean arterial pressure = DBP + one-third of pulse pressure
    out["mean_arterial_pressure"] = out["triage_vital_dbp"] + (out["pulse_pressure"] / 3)

    # 2) Non-blood pressure red flags
    out["is_tachypneic"] = (out["triage_vital_rr"] > 20).astype(int)           # Fast breathing
    out["is_hypoxic"] = (out["triage_vital_o2"] < 92).astype(int)              # Low oxygen
    out["is_febrile"] = (out["triage_vital_temp"] >= 100.4).astype(int)        # Fever
    out["is_bradycardic"] = (out["triage_vital_hr"] < 60).astype(int)          # Slow heart rate
    out["is_hyperglycaemic"] = (out["triage_glucose"] > 180).astype(int) # Poorly controlled/Physiologically stressed
    out["is_hypothermic"] = (out["triage_vital_temp"] < 96.8).astype(int)      # Hypothermic

    # 3) Combined red flags
    # Combined respiratory red flag: 1 if the patient is hypoxic OR tachypneic
    out["resp_distress"] = ((out["is_tachypneic"] == 1) | (out["is_hypoxic"] == 1)).astype(int)

    # 4) Ratios
    # Oxygen saturation to respiratory rate ratio
    out["spo2_rr_ratio"] = out["triage_vital_o2"] / out["triage_vital_rr"]

    # Pulse pressure
    out["pulse_pressure"] = out["triage_vital_sbp"] - out["triage_vital_dbp"]

    # Counting the number of active red flags
    red_flag_cols = ["is_tachypneic", "is_hypoxic", "is_febrile", "is_bradycardic", "is_hyperglycaemic", "is_hypothermic"]
    out["red_flag_count"] = out[red_flag_cols].sum(axis=1)


    return out

X_train_fe = add_clinical_features(X_train)
X_test_fe = add_clinical_features(X_test)
print("Features after engineering:", X_train_fe.shape[1])
X_train_fe.head()

Features after engineering: 220


,triage_vital_hr,triage_vital_sbp,triage_vital_dbp,triage_vital_rr,triage_vital_o2,triage_vital_o2_device,triage_vital_temp,triage_glucose,cc_abdominalcramping,cc_abdominaldistention,...,spo2_rr_ratio,mean_arterial_pressure,is_tachypneic,is_hypoxic,is_febrile,is_bradycardic,is_hyperglycaemic,is_hypothermic,resp_distress,red_flag_count
35369,104.0,120.0,71.0,22.0,98.0,1.0,98.2,137.0,0.0,0.0,...,4.454545,87.333333,1,0,0,0,0,0,1,1
52043,78.0,115.0,76.0,18.0,96.0,0.0,98.4,102.0,0.0,0.0,...,5.333333,89.000000,0,0,0,0,0,0,0,0
13610,96.0,119.0,78.0,18.0,94.0,0.0,98.1,108.0,0.0,0.0,...,5.222222,91.666667,0,0,0,0,0,0,0,0
54796,89.0,128.0,93.0,16.0,98.0,0.0,97.7,108.0,0.0,0.0,...,6.125000,104.666667,0,0,0,0,0,0,0,0
11096,89.0,113.0,78.0,18.0,98.0,0.0,98.1,92.0,0.0,0.0,...,5.444444,89.666667,0,0,0,0,0,0,0,0


---
##**Section 4: Model 1 - Random Forest Model**

This section trains an initial random-forest classifier using the engineered clinical featured. A random forest combines predictions from multiple decision trees, allowing it to capture more complex and non-linear relationships between patient characteristics and ESI level than a single decision tree or logistic-regression model.

A fixed random seed of 42 ensures reproducibility, while parallel processing reduces training time. The initial model is evaluated using macro F1 and its feature-importance scores are also examined to identify the clinical variables that most influenced its predictions.

####**4.1 Random Forest Model Training**

In [9]:
# Creating the initial random-forest model
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)

# Training using engineered features
rf.fit(X_train_fe, y_train)

# Evaluating the model's performance
rf_f1 = f1_score(y_test, rf.predict(X_test_fe), average="macro")

# Ranking feature importance from most to least important
rf_feature_importance = pd.Series(
    rf.feature_importances_,
    index=X_train_fe.columns,
    name = "Feature Importance"
).sort_values(ascending=False)

# Display the 15 most important features
rf_feature_importance.head(15).to_frame()

,Feature Importance
mean_arterial_pressure,0.069636
shock_index,0.067841
triage_vital_sbp,0.064937
triage_glucose,0.062263
triage_vital_hr,0.061255
triage_vital_dbp,0.061109
pulse_pressure,0.059957
triage_vital_temp,0.055113
spo2_rr_ratio,0.046950
cc_strokealert,0.037093


####**4.2 Encoding Categorical Features**

This section investigates whether adding demographic information improves the random forest’s ability to predict ESI level. Age and gender are already numerical, while ethnicity and race must be converted from text categories into a format the model can process.

One-hot encoding creates a separate binary column for each ethnicity and race category. A value of 1 indicates that the patient belongs to that category, while 0 indicates that they do not.

The random forest is retrained using the expanded feature set and compared with the demographics-free model from Section 4. Although demographic variables may improve predictive performance, their use must be evaluated carefully because the model could learn historical demographic differences or biases in triage decisions.

In [10]:
# One-hot encoding race and ethnicity
demo_1hot = pd.get_dummies(df[["ethnicity", "race"]], prefix=["eth", "race"], dtype=int)
print("New one-hot columns:", list(demo_1hot.columns))

def add_demographics(X_fe):
    """Bolt the encoded demographics onto an existing feature frame (aligned by row)."""
    rows = X_fe.index
    extra = demo_1hot.loc[rows].copy()
    extra["age"] = df.loc[rows, "age"]         # numeric already
    extra["gender"] = df.loc[rows, "gender"].map({"female":0, "Male": 1})
    return pd.concat([X_fe, extra], axis=1)

X_train_plus = add_demographics(X_train_fe)
X_test_plus = add_demographics(X_test_fe)
print("Features WITHOUT demographics:", X_train_fe.shape[1])
print("Features WITH    demographics:", X_train_plus.shape[1])
X_train_plus.filter(like="race_").head()

New one-hot columns: ['eth_Hispanic or Latino', 'eth_Non-Hispanic', 'eth_Patient Refused', 'eth_Unknown', 'race_American Indian or Alaska Native', 'race_Asian', 'race_Black or African American', 'race_Native Hawaiian or Other Pacific Islander', 'race_Other', 'race_Patient Refused', 'race_Unknown', 'race_White or Caucasian']
Features WITHOUT demographics: 220
Features WITH    demographics: 234


,race_American Indian or Alaska Native,race_Asian,race_Black or African American,race_Native Hawaiian or Other Pacific Islander,race_Other,race_Patient Refused,race_Unknown,race_White or Caucasian
35369,0,0,0,0,1,0,0,0
52043,0,0,0,0,0,0,0,1
13610,0,0,0,0,0,0,0,1
54796,0,0,1,0,0,0,0,0
11096,0,0,0,0,0,0,0,1


####**4.3 Retraining Random Forest Model with Encoded Demographics**

In [11]:
# Retrain the random forest with encoded demographics
rf_demo = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_demo.fit(X_train_plus, y_train)

rf_demo_f1 = f1_score(
    y_test,
    rf_demo.predict(X_test_plus),
    average="macro"
)

print("Random forest without demographics:", round(rf_f1, 3))
print("Random forest with demographics:", round(rf_demo_f1, 3))
print("Change in macro F1:", round(rf_demo_f1 - rf_f1, 3))


Random forest without demographics: 0.38
Random forest with demographics: 0.388
Change in macro F1: 0.008


###**Section 4 Observations**

Adding age, gender, race and ethnicity increased the feature set from 220 to 234 variables. The random forest's Macro F1 score showed a slight improvement of 0.008 going from 0.380 to 0.388. This suggests that demographic information provided a small amount of additional predictive value, but the improvement was marginal. It may not be large enough to justify the ethical and equity risks of including sensitive demographic variables.

---
##**Section 5: Model 2 - Gradient Boosting Model**

This section trains a histogram-based gradient-boosting classifier using the engineered clinical features. Unlike a random forest, which builds trees independently, gradient boosting builds trees sequentially, with each new tree attempting to correct errors made by the previous trees. This allows the model to capture complex, non-linear relationships between patient characteristics and ESI level. Class balancing increases attention to less common ESI categories, while a fixed random seed of 42 supports reproducibility. The model is evaluated using macro F1 so that performance across all five ESI levels is weighted equally.

In [12]:
# Training a gradient-boosting model. No scaling needed.
hgb = HistGradientBoostingClassifier(
    max_depth=6, learning_rate=0.1, max_iter=300,
    class_weight="balanced", random_state=42)
hgb.fit(X_train_fe, y_train)
hgb_f1 = f1_score(y_test, hgb.predict(X_test_fe), average="macro")
print("Gradient Boosting macro-F1:", round(hgb_f1, 3))

Gradient Boosting macro-F1: 0.399


---
##**Section 6: Model 3 - MLP Model**

This section trains a small multilayer perceptron (MLP) neural network using the engineered clinical features. The MLP uses interconnected layers of neurons to learn complex, non-linear relationships between patient characteristics and ESI level. The features are standardised before training because MLP models are sensitive to differences in input scales. Regularisation is applied to reduce overfitting, while a fixed random seed of 42 supports reproducibility. The model is evaluated using macro F1 to give equal importance to all five ESI levels.

In [13]:
# Training a small neural network on SCALED features
mlp = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-3, max_iter=500, random_state=42))
mlp.fit(X_train_fe, y_train)
mlp_f1 = f1_score(y_test, mlp.predict(X_test_fe), average="macro")
print("Small MLP macro-F1:", round(mlp_f1, 3))

Small MLP macro-F1: 0.464


---
##**Section 7: Hyperparameter Tuning with Cross-Validation**

This section uses randomized hyperparameter tuning to improve the random-forest, gradient-boosting and MLP models. Hyperparameters control how each model learns, including the number and depth of trees, the learning rate, regularisation strength and neural-network structure.

'RandomizedSearchCV' tests randomly selected combinations using three-fold cross-validation. In each trial, the training data is divided into three folds: two folds train the model and the remaining fold validates it. This process is repeated so that every fold is used for validation.

Macro F1 is used to select the best combination because it gives equal importance to every ESI class, including the less common urgent categories. The test set remains separate during tuning and is used only to evaluate the final selected model.



###**7.1 Random Forest Tuning**

The following Random Forest hyperparameters will be tuned:

| Hyperparameter | Meaning |
|---|---|
| **`n_estimators`** | Number of trees in the forest.<br>More trees may produce steadier predictions but increase training time. |
| **`max_depth`** | Maximum depth of each tree.<br>Greater depth captures more detail but may increase overfitting. |
| **`min_samples_leaf`** | Minimum number of patients required in a final leaf.<br>Larger values create smoother, less complex trees. |
| **`max_features`** | Number of features considered at each split.<br>Using fewer features increases variation among the trees. |
| **`class_weight="balanced"`** | Gives greater importance to rare classes, including ESI Level 1.<br>This setting was fixed rather than varied during tuning. |


In [16]:
#searches random-forest settings with 3-fold CV and
# keeps the combination that scores best on macro-F1.

param_dist = {
    "n_estimators": [100, 200],
    "max_depth": [None, 6, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None],
}

# Search randomly selected combinations
search = RandomizedSearchCV(
    RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=1
    ),
    param_distributions=param_dist,
    n_iter=8,
    cv=3,
    scoring="f1_macro",
    random_state=42,
    n_jobs=-1
)

search.fit(X_train_fe, y_train)
print(f"Best parameters: {search.best_params_}")
print(f"Best macro F1: {search.best_score_}")

# Saving and testing best random forest model
rf_tuned = search.best_estimator_
rf_tuned_f1 = f1_score(
    y_test,
    rf_tuned.predict(X_test_fe),
    average="macro"
)

print(f"Test set macro F1: {rf_tuned_f1}")



Best parameters: {'n_estimators': 200, 'min_samples_leaf': 4, 'max_features': None, 'max_depth': None}
Best macro F1: 0.4543027065795628
Test set macro F1: 0.47753484627283066


###**7.2 Gradient-Boosting Model Tuning**

The following Gradient-Boosting Model parameters will be tuned:
| Hyperparameter | Meaning |
|---|---|
| **`learning_rate`** | Size of the correction made during each boosting round.<br>Smaller values allow the model to learn more gradually. |
| **`max_iter`** | Number of sequential boosting rounds.<br>More rounds may improve learning but increase training time. |
| **`max_depth`** | Maximum depth of each tree.<br>Shallower trees generally reduce the risk of overfitting. |
| **`min_samples_leaf`** | Minimum number of patients required in each final leaf.<br>Larger values create smoother, less complex trees. |
| **`class_weight="balanced"`** | Gives greater importance to rare ESI classes, including ESI Level 1.<br>This setting was fixed rather than varied during tuning. |

In [14]:
# searches gradient-boosting settings with 3-fold CV and
# keeps the combination that scores best on macro-F1.

hgb_param_dist = {
    "learning_rate": [0.05, 0.1, 0.2],
    "max_iter": [100, 200, 300],
    "max_depth": [None, 3, 10],
    "min_samples_leaf": [10, 20],
}

# Randomly test eight combinations using three-fold CV
hgb_search = RandomizedSearchCV(
    HistGradientBoostingClassifier(
        class_weight="balanced",
        random_state=42,
    ),
    param_distributions=hgb_param_dist,
    n_iter=8,
    cv=3,
    scoring="f1_macro",
    random_state=42,
    n_jobs=-1
)

hgb_search.fit(X_train_fe, y_train)
print(f"Best parameters: {hgb_search.best_params_}")
print(f"Best macro F1: {hgb_search.best_score_}")

# Saving and test best gradient-boosting model
hgb_tuned = hgb_search.best_estimator_
hgb_tuned_f1 = f1_score(
    y_test,
    hgb_tuned.predict(X_test_fe),
    average="macro"
)

print(f"Test set macro F1: {hgb_tuned_f1}")

Best parameters: {'min_samples_leaf': 20, 'max_iter': 100, 'max_depth': None, 'learning_rate': 0.1}
Best macro F1: 0.44085865610114155
Test set macro F1: 0.4412727148194934


###**7.3 MLP Tuning**

The following MLP model parameters were tuned:

| Hyperparameter | Meaning |
|---|---|
| **`hidden_layer_sizes`** | Number and size of the neural network’s hidden layers.<br>Larger networks have greater learning capacity but require more <br>computation and may overfit. |
| **`alpha`** | Regularisation strength.<br>Higher values constrain the model and may reduce overfitting. |
| **`learning_rate_init`** | Initial size of the adjustments made while the network learns.<br>Larger values may learn faster but can make training less stable. |
| **`max_iter`** | Maximum number of training iterations before the model stops.<br>This was fixed at 500 rather than varied during tuning. |

In [14]:
# Searches mlp settings with 3-fold CV and
# Keeps the combination that scores best on macro-F1.

mlp_pipeline = make_pipeline(
    StandardScaler(),
    MLPClassifier(max_iter=500, random_state=42)
)
mlp_param_dist = {
    "mlpclassifier__hidden_layer_sizes": [(64, 32), (128, 64)],
    "mlpclassifier__alpha": [1e-3, 1e-4],
    "mlpclassifier__learning_rate_init": [0.001, 0.0005]
}

# Randomly testing eight combinations using three-fold CV
mlp_search = RandomizedSearchCV(
    mlp_pipeline,
    param_distributions=mlp_param_dist,
    n_iter=8,
    cv=3,
    scoring="f1_macro",
    random_state=42,
    n_jobs=-1,
    verbose=2
)

mlp_search.fit(X_train_fe, y_train)
print(f"Best parameters: {mlp_search.best_params_}")
print(f"Best macro F1: {mlp_search.best_score_}")

# Saving and testing best mlp
mlp_tuned = mlp_search.best_estimator_
mlp_tuned_f1 = f1_score(
    y_test,
    mlp_tuned.predict(X_test_fe),
    average="macro"
)

print(f"Test macro F1:", (mlp_tuned_f1))


Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best parameters: {'mlpclassifier__learning_rate_init': 0.0005, 'mlpclassifier__hidden_layer_sizes': (64, 32), 'mlpclassifier__alpha': 0.001}
Best macro F1: 0.4468918940327869
Test macro F1: 0.47007825980633405


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


---
## **Section 8: Six-Axis Benchmark**

This section compares the logistic-regression baseline and the three tuned complex models across six quantitative axes:

1. **Accuracy** - the proportion of all patients classified correctly.
2. **Macro precision** - average precision across the five ESI levels.
3. **Macro recall** - average recall across the five ESI levels.
4. **Macro F1** - the average F1 score across the five ESI levels.
5. **Training time** - the time required to fit the selected model.
6. **Inference time** - the average time required to predict one patient's ESI level.

Interpretability is included as a seventh qualitative axis. It assesses whether a single prediction can be explained in under one minute and identifies the method required. ESI Level 1 recall is reported as an additional clinical-safety metric because failure to identify these patients could delay life-saving treatment.

The reported training times cover fitting the selected models and do not include the earlier hyperparameter searches. Inference time represents the average per-patient time calculated from batch prediction of the test set.

In [19]:
# Rebuilding the tuned Random Forest using the best parameters
# previously identified by RandomizedSearchCV

rf_tuned = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=4,
    max_features=None,
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_tuned.fit(X_train_fe, y_train)
rf_tuned_predictions = rf_tuned.predict(X_test_fe)
rf_tuned_f1 = f1_score(y_test, rf_tuned_predictions, average="macro",zero_division=0)
print("Tuned Random Forest macro-F1:", round(rf_tuned_f1, 3))

# Rebuilding the tuned Gradient Boosting model using the best parameters
# previously identified by RandomizedSearchCV

hgb_tuned = HistGradientBoostingClassifier(
    min_samples_leaf=20,
    max_iter=100,
    max_depth=None,
    learning_rate=0.1,
    class_weight="balanced",
    random_state=42
)

hgb_tuned.fit(X_train_fe, y_train)
hgb_tuned_predictions = hgb_tuned.predict(X_test_fe)
hgb_tuned_f1 = f1_score(
    y_test,
    hgb_tuned_predictions,
    average="macro",
    zero_division=0
)
print("Tuned Gradient Boosting macro-F1:", round(hgb_tuned_f1, 3))

# Rebuilding the tuned MLP using the best parameters
# previously identified by RandomizedSearchCV

mlp_tuned = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(64, 32),
        alpha=0.001,
        learning_rate_init=0.0005,
        max_iter=500,
        random_state=42
    )
)

mlp_tuned.fit(X_train_fe, y_train)
mlp_tuned_predictions = mlp_tuned.predict(X_test_fe)
mlp_tuned_f1 = f1_score(
    y_test,
    mlp_tuned_predictions,
    average="macro",
    zero_division=0
)

print("Tuned MLP macro-F1:", round(mlp_tuned_f1, 3))

Tuned Random Forest macro-F1: 0.478
Tuned Gradient Boosting macro-F1: 0.441
Tuned MLP macro-F1: 0.47


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [25]:
# Evaluating all models with full six-axis benchmark

scores = {
    "Baseline (LogReg)": baseline_f1,
    "Random Forest": rf_f1,
    "Random Forest (tuned)": rf_tuned_f1,
    "Gradient Boosting": hgb_f1,
    "Gradient Boosting (tuned)": hgb_tuned_f1,
    "MLP": mlp_f1,
    "MLP (tuned)": mlp_tuned_f1
}

pd.Series(scores).sort_values(ascending=False).round(3).to_frame("macro_F1")

def benchmark_model(
    name,
    model,
    X_train_set,
    X_test_set,
    interpretability
):
    # AXIS 5 - Training time
    start = time.perf_counter()
    model.fit(X_train_set, y_train)
    train_time = time.perf_counter() - start

    # AXIS 6 - Inference time
    start = time.perf_counter()
    predictions = model.predict(X_test_set)
    inference_time = time.perf_counter() - start

    # Placing the benchmark results into one dictionary
    return {
        "Model": name,

        # AXIS 1: Overall proportion of correct predictions
        "Accuracy": accuracy_score(y_test,predictions),

        # AXIS 2: Average precision across all five ESI levels
        "Macro Precision": precision_score(y_test,predictions,average="macro",zero_division=0),

        # AXIS 3: Average recall across all five ESI levels
        "Macro Recall": recall_score(y_test, predictions,average="macro",zero_division=0),

        # Additional clinical metric:
        # Proportion of actual ESI Level 1 patients detected
        "ESI Level 1 Recall": recall_score(y_test,predictions,labels=[1],average=None,zero_division=0)[0],

        # AXIS 4: Balanced performance across all five ESI levels
        "Macro F1": f1_score(y_test,predictions,average="macro",zero_division=0),

        # AXIS 5: Model training time
        "Training time (s)": train_time,

        # AXIS 6: Prediction time for one patient in milliseconds
        "Inference time (ms/patient)": (inference_time * 1000 / X_test_set.shape[0]),

        # AXIS 7: Interpretability
        "Interpretability": interpretability
    }


In [26]:
# Benchmarking results
benchmark_results = [benchmark_model("Baseline (LogReg)",baseline,X_train,X_test,
        "High - coefficient contributions can be explained in under one minute"
    ),

    benchmark_model("Random Forest (tuned)",rf_tuned,X_train_fe,X_test_fe,
        "Moderate - feature importance or SHAP can be used"
    ),

    benchmark_model("Gradient Boosting (tuned)",hgb_tuned,X_train_fe,X_test_fe,
        "Moderate - requires a SHAP explanation"
    ),

    benchmark_model("Small MLP (tuned)",mlp_tuned,X_train_fe,X_test_fe,
        "Low - difficult to explain without an additional explanation method"
    )
]

# Converting the benchmark results into a table
benchmark_df = pd.DataFrame(benchmark_results)

# Rounding the numerical results to three decimal places
numeric_columns = benchmark_df.select_dtypes(include="number").columns
benchmark_df[numeric_columns] = benchmark_df[numeric_columns].round(3)

# Displaying the completed benchmark table
benchmark_df

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


,Model,Accuracy,Macro Precision,Macro Recall,ESI Level 1 Recall,Macro F1,Training time (s),Inference time (ms/patient),Interpretability
0,Baseline (LogReg),0.667,0.594,0.463,0.250,0.495,7.202,0.003,High - coefficient contributions can be explai...
1,Random Forest (tuned),0.626,0.470,0.495,0.312,0.478,239.360,0.049,Moderate - feature importance or SHAP can be used
2,Gradient Boosting (tuned),0.589,0.425,0.566,0.312,0.441,8.980,0.026,Moderate - requires a SHAP explanation
3,Small MLP (tuned),0.634,0.533,0.444,0.188,0.470,232.699,0.003,Low - difficult to explain without an addition...


In [27]:
# Saving models
joblib.dump(rf_tuned, "w7_random_forest.joblib")
joblib.dump(hgb_tuned, "w7_gradient_boosting.joblib")
joblib.dump(mlp_tuned, "w7_mlp.joblib")
print("Saved: w7_random_forest.joblib, w7_gradient_boosting.joblib, w7_mlp.joblib ✅")

# To reload later
#   rf_tuned = joblib.load("w7_random_forest.joblib")

Saved: w7_random_forest.joblib, w7_gradient_boosting.joblib, w7_mlp.joblib ✅


##**Section 9: Conclusion and Preliminary Recommendation**

The logistic-regression baseline should remain the preferred model because none of the tuned complex models provided enough benefit to justify their added complexity.

Logistic regression achieved the highest accuracy (0.667), macro precision (0.594) and macro F1 (0.495). It trained in 7.202 seconds, predicted in approximately 0.003 milliseconds per patient and offered the clearest explanation method. The tuned random forest was the strongest complex model, but its macro F1 remained lower at 0.478, while training took approximately 239 seconds. Although it improved macro recall and ESI Level 1 recall, these gains require confirmation using a larger sample of critically ill patients. Gradient boosting achieved the highest macro recall (0.566), but produced lower accuracy, precision and macro F1. The tuned MLP also failed to outperform logistic regression and did not fully converge.

Therefore, logistic regression is recommended for further validation as an explainable clinical prototype, with the tuned random forest retained as a challenger model. This choice does not resolve class imbalance, under-triage, calibration, demographic bias, dataset shift or the need for external validation. Clinicians must retain final decision-making authority.

##**Clinical question:**
*In one sentence, explain to Martina Griffith why a 0.01 F1 gain might still not be worth deploying.*

Even a 0.01 F1 improvement would not justify deployment if it failed to produce a clinically meaningful reduction in harmful under-triage while increasing explanation, validation and operational burdens.




